# LeetCode 524 — Longest Word in Dictionary through Deleting bằng Python

Link đề: https://leetcode.com/problems/longest-word-in-dictionary-through-deleting/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, String, Sorting

Skeleton LeetCode:

```python
class Solution:
    def findLongestWord(self, s: str, dictionary: List[str]) -> str:
        ...
```

Ghi chú: các ví dụ 3, 4, 5 trong mục 2 do notebook tự đặt để minh hoạ biên và luật hoà (cùng độ dài), đáp án đã kiểm chứng bằng code; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong `dictionary`, tìm từ **dài nhất** có thể tạo ra từ `s` bằng cách **xoá một số ký tự** (tức là từ đó là **dãy con / subsequence** của `s`); nếu nhiều từ cùng độ dài thì lấy từ **nhỏ nhất theo thứ tự từ điển**; không có thì trả về chuỗi rỗng `""`.
2. Nắm các công cụ Python: so sánh chuỗi bằng `<`, `sorted(..., key=...)` với khoá là tuple, vòng `for` duyệt chuỗi, `all(...)` với `iter(...)`, `dict`/`defaultdict`, `bisect`.
3. Đi từ **liệt kê mọi dãy con của `s` (2ⁿ)** → **kiểm tra từng từ bằng hai con trỏ** → **sắp xếp theo khoá `(-độ dài, từ)` rồi trả về từ hợp lệ đầu tiên** → **bảng "ký tự kế tiếp" nhảy cóc** → **quét `s` một lần với các "xô chờ"**.
4. Hiểu mấu chốt: đừng sinh mọi dãy con của `s`; hãy **đảo vai trò** — lấy từng từ trong từ điển rồi hỏi "nó có là dãy con của `s` không?" (kiểm tra tham lam `O(|s|)`), và dùng **một khoá so sánh duy nhất** `(-len, word)` cho cả hai tiêu chí.
5. Tránh các lỗi kinh điển: nhầm dãy con với chuỗi con (substring), nhầm với "đủ ký tự" (tập/đếm chữ cái), quên luật hoà theo từ điển, dùng `max(..., key=len)` (không đúng thứ tự từ điển), và lỗi `IndexError` khi con trỏ đã hết từ.

## 2. Đề bài & ví dụ

Cho chuỗi `s` và mảng chuỗi `dictionary`. Trả về **chuỗi dài nhất trong `dictionary`** có thể tạo thành bằng cách **xoá một số ký tự** của `s`. Nếu có nhiều đáp án, trả về từ **dài nhất và nhỏ nhất theo thứ tự từ điển**. Nếu không có đáp án nào, trả về chuỗi rỗng.

(Xoá một số ký tự mà **không đổi thứ tự** các ký tự còn lại — chính là tạo ra một **dãy con** của `s`. Có thể xoá 0 ký tự.)

**Ví dụ 1**

- Input: `s = "abpcplea"`, `dictionary = ["ale","apple","monkey","plea"]`
- Output: `"apple"`
- Giải thích: `apple` và `ale`, `plea` đều là dãy con của `s`; `apple` dài nhất (5 ký tự). `monkey` không tạo được.

**Ví dụ 2**

- Input: `s = "abpcplea"`, `dictionary = ["a","b","c"]`
- Output: `"a"`
- Giải thích: cả ba đều tạo được, cùng độ dài 1; `"a"` nhỏ nhất theo từ điển.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `s = "bab"`, `dictionary = ["ba","ab","a","b"]`
- Output: `"ab"`
- Giải thích: `"ba"` và `"ab"` đều là dãy con độ dài 2; `"ab" < "ba"`.

**Ví dụ 4 (tự đặt)**

- Input: `s = "abc"`, `dictionary = ["d","cab"]`
- Output: `""`
- Giải thích: `d` không có trong `s`; `cab` đủ ký tự nhưng sai thứ tự nên không phải dãy con.

**Ví dụ 5 (tự đặt)**

- Input: `s = "abc"`, `dictionary = ["abc","abc","ab"]`
- Output: `"abc"` (từ điển có thể chứa từ lặp lại).

**Ràng buộc:**

- `1 <= s.length <= 1000`
- `1 <= dictionary.length <= 1000`
- `1 <= dictionary[i].length <= 1000`
- `s` và `dictionary[i]` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
from typing import List
from collections import defaultdict
from bisect import bisect_right
from itertools import combinations
import random
import time

examples = [
    ("abpcplea", ["ale", "apple", "monkey", "plea"], "apple"),
    ("abpcplea", ["a", "b", "c"], "a"),
    ("bab", ["ba", "ab", "a", "b"], "ab"),
    ("abc", ["d", "cab"], ""),
    ("abc", ["abc", "abc", "ab"], "abc"),
]
for s, dictionary, expected in examples:
    print(f"s={s!r}, dictionary={dictionary} -> kỳ vọng {expected!r}")

s='abpcplea', dictionary=['ale', 'apple', 'monkey', 'plea'] -> kỳ vọng 'apple'
s='abpcplea', dictionary=['a', 'b', 'c'] -> kỳ vọng 'a'
s='bab', dictionary=['ba', 'ab', 'a', 'b'] -> kỳ vọng 'ab'
s='abc', dictionary=['d', 'cab'] -> kỳ vọng ''
s='abc', dictionary=['abc', 'abc', 'ab'] -> kỳ vọng 'abc'


## 3. Kiến thức nền

### 3.1. Dãy con (subsequence) khác chuỗi con (substring)

- **Chuỗi con (substring):** một đoạn **liền nhau**, ví dụ các chuỗi con của `"abc"` có `"ab"`, `"bc"` nhưng không có `"ac"`.
- **Dãy con (subsequence):** chọn một số ký tự **giữ nguyên thứ tự** nhưng **có thể cách nhau**; `"ac"` là dãy con của `"abc"` (xoá `b`).

Toán tử `in` của Python (`"ab" in "abc"`) kiểm tra **chuỗi con**, không phải dãy con. Bài này cần dãy con.

In [2]:
s = "abpcplea"
print("'plea' là chuỗi con của s?  ", "plea" in s)      # liền nhau: s có 'plea' ở cuối
print("'apple' là chuỗi con của s? ", "apple" in s)     # False: a-p-p-l-e không liền nhau
print("nhưng 'apple' là dãy con: a(0) p(2) p(4) l(5) e(6)")

'plea' là chuỗi con của s?   True
'apple' là chuỗi con của s?  False
nhưng 'apple' là dãy con: a(0) p(2) p(4) l(5) e(6)


### 3.2. Kiểm tra dãy con bằng hai con trỏ

Để kiểm tra từ `t` có là dãy con của `s` không: dùng con trỏ `i` trên `t` (ký tự `t` cần ghép tiếp theo) và quét `s` từ trái sang phải; mỗi khi ký tự `s[j] == t[i]` thì `i += 1`. Cuối cùng nếu `i == len(t)` tức đã ghép hết `t`. Cách ghép **tham lam** (ghép sớm nhất có thể) là đủ, ta sẽ chứng minh ở mục 10.

In [3]:
def is_subseq(t, s):
    i = 0
    for ch in s:
        if i < len(t) and ch == t[i]:
            i += 1
    return i == len(t)

print(is_subseq("apple", "abpcplea"), is_subseq("monkey", "abpcplea"), is_subseq("cab", "abc"), is_subseq("", "abc"))

True False False True


### 3.3. So sánh chuỗi theo thứ tự từ điển

Python so sánh chuỗi bằng `<`, `>`, `==` theo **thứ tự từ điển** (so từng ký tự từ trái, ký tự nhỏ hơn thì chuỗi nhỏ hơn; nếu một chuỗi là tiền tố của chuỗi kia thì chuỗi ngắn hơn nhỏ hơn). Chữ thường `a < b < ... < z`.

In [4]:
print("ab" < "ba", "apple" < "apply", "app" < "apple", "b" < "ab")
print(sorted(["plea", "ale", "apple", "ab"]))

True True True False
['ab', 'ale', 'apple', 'plea']


### 3.4. Sắp xếp bằng `key` là một tuple

`sorted(words, key=...)` sắp theo giá trị `key(w)` của từng phần tử. Nếu `key` trả về **tuple**, Python so sánh theo từng phần tử của tuple: so phần tử đầu, hoà thì so phần tử thứ hai... Với khoá `(-len(w), w)`: số **âm** của độ dài làm từ **dài hơn** đứng trước; hoà độ dài thì từ **nhỏ hơn theo từ điển** đứng trước. Đó chính xác là thứ tự "tốt nhất" mà đề yêu cầu.

In [5]:
words = ["ale", "apple", "monkey", "plea", "a", "b", "ab", "ba"]
print(sorted(words, key=lambda w: (-len(w), w)))
print(min(words, key=lambda w: (-len(w), w)), "<- min theo khoá chính là phần tử 'tốt nhất'")

['monkey', 'apple', 'plea', 'ale', 'ab', 'ba', 'a', 'b']
monkey <- min theo khoá chính là phần tử 'tốt nhất'


### 3.5. `all(...)` với `iter(...)`: mẹo kiểm tra dãy con một dòng

Một iterator "nhớ vị trí đã đọc". `c in it` đọc tiếp `it` cho tới khi gặp `c` (và **tiêu thụ** đến chỗ đó). Do vậy `all(c in it for c in t)` với `it = iter(s)` kiểm tra đúng dãy con. Gọn nhưng hơi "ma thuật", nên bài này ta viết vòng `for` tường minh; nhưng đáng biết.

In [6]:
def is_subseq_iter(t, s):
    it = iter(s)
    return all(c in it for c in t)

print(is_subseq_iter("apple", "abpcplea"), is_subseq_iter("cab", "abc"))
it = iter("abpcplea")
print("a" in it, "p" in it, "p" in it, list(it))   # iterator đã đi tới sau chữ p thứ hai

True False
True True True ['l', 'e', 'a']


### 3.6. `dict` / `defaultdict` làm "xô" và `bisect`

`defaultdict(list)` tự tạo list rỗng khi gặp khoá mới — tiện để **gom** các phần tử theo một khoá (ở cách 5 ta gom từ theo "ký tự đang chờ"). `bisect_right(lst, x)` trả về vị trí chèn `x` vào list đã sắp xếp (bên phải các phần tử bằng `x`) bằng tìm kiếm nhị phân `O(log n)` — dùng ở mục biến thể.

In [7]:
groups = defaultdict(list)
for w in ["ale", "apple", "monkey", "plea"]:
    groups[w[0]].append(w)
print(dict(groups))

pos = [1, 4, 7, 9]
print(bisect_right(pos, 4), bisect_right(pos, 5), bisect_right(pos, 0))   # chỉ số phần tử đầu tiên > x

{'a': ['ale', 'apple'], 'm': ['monkey'], 'p': ['plea']}
2 2 0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `s = "abpcplea"` và từ `"apple"`. Cách của con người: **gạch dưới trong `s` các chữ cái khớp lần lượt a, p, p, l, e**.

| Ký tự của `s` | `a` | `b` | `p` | `c` | `p` | `l` | `e` | `a` |
|---|---|---|---|---|---|---|---|---|
| chỉ số | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
| ghép với | `a` (1) | bỏ | `p` (2) | bỏ | `p` (3) | `l` (4) | `e` (5) | thừa |

Ghép đủ 5 ký tự → `"apple"` là dãy con. Với `"monkey"`: ký tự đầu `m` không có trong `s` → dừng. Trực giác: **mỗi từ cần "đọc" `s` một lần từ trái sang phải**, không cần quay lại.

Có 4 từ, ta kiểm tra từng từ và ghi **độ dài**:

| Từ | Dãy con của `s`? | Độ dài |
|---|---|---|
| `ale` | có (a0, l5, e6) | 3 |
| `apple` | có | **5** |
| `monkey` | không | — |
| `plea` | có (p2, l5, e6, a7) | 4 |

Từ dài nhất hợp lệ: `apple` (5). Nếu có hai từ hợp lệ cùng độ dài 5 ta lấy cái nhỏ hơn theo từ điển.

In [8]:
s = "abpcplea"
def match_positions(t, s):
    """Trả về danh sách chỉ số trong s mà mỗi ký tự của t được ghép (tham lam), hoặc None."""
    out, i = [], 0
    for j, ch in enumerate(s):
        if i < len(t) and ch == t[i]:
            out.append(j)
            i += 1
    return out if i == len(t) else None

for t in ["ale", "apple", "monkey", "plea"]:
    print(f"{t:<7}", match_positions(t, s))

ale     [0, 5, 6]
apple   [0, 2, 4, 5, 6]
monkey  None
plea    [2, 5, 6, 7]


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi dãy con của `s`

### 5.1. Ý tưởng

Dựng **tập mọi dãy con của `s`**, rồi với mỗi từ trong từ điển, kiểm tra nó có nằm trong tập đó không; trong các từ có mặt, chọn từ dài nhất / nhỏ nhất từ điển.

Dựng tập bằng cách thêm từng ký tự: với mỗi `ch` của `s`, mọi dãy con đã có `x` sinh thêm `x + ch` (lấy ký tự đó) bên cạnh `x` (không lấy). Mỗi ký tự làm số dãy con **gấp đôi** (tối đa `2^n`).

In [9]:
def all_subsequences(s):
    subs = {""}
    for ch in s:
        subs |= {x + ch for x in subs}      # vế phải tính trước: x đã có -> x + ch
    return subs

print(len(all_subsequences("abc")), sorted(all_subsequences("abc")))
print("abpcplea:", len(all_subsequences("abpcplea")), "dãy con khác nhau (tối đa 2^8 =", 2 ** 8, ")")

8 ['', 'a', 'ab', 'abc', 'ac', 'b', 'bc', 'c']
abpcplea: 223 dãy con khác nhau (tối đa 2^8 = 256 )


In [10]:
def brute_force(s, dictionary):
    subs = all_subsequences(s)
    best = ""
    for w in dictionary:
        if w in subs:
            if len(w) > len(best) or (len(w) == len(best) and w < best):
                best = w
    return best

for s, dictionary, expected in examples:
    got = brute_force(s, dictionary)
    print(f"{s!r}, {dictionary} -> {got!r}")
    assert got == expected

'abpcplea', ['ale', 'apple', 'monkey', 'plea'] -> 'apple'
'abpcplea', ['a', 'b', 'c'] -> 'a'
'bab', ['ba', 'ab', 'a', 'b'] -> 'ab'
'abc', ['d', 'cab'] -> ''
'abc', ['abc', 'abc', 'ab'] -> 'abc'


### 5.2. Độ phức tạp và nút thắt

- Time: `O(2^n · n)` (sinh tới `2^n` chuỗi, mỗi chuỗi dài tới `n`), Space cũng cỡ `O(2^n · n)`.
- Với `n = 1000` thì `2^1000` là con số khổng lồ — không thể.

Nút thắt: ta **sinh quá nhiều thứ không ai hỏi tới** (hầu hết dãy con của `s` không phải từ nào trong từ điển). Hãy xem số dãy con tăng nhanh thế nào:

In [11]:
for n in (4, 8, 12, 16):
    t0 = time.perf_counter()
    cnt = len(all_subsequences("abcdefghijklmnop"[:n]))
    print(f"n={n:2d}: {cnt:6d} dãy con, {1000 * (time.perf_counter() - t0):7.2f} ms")

n= 4:     16 dãy con,    0.01 ms
n= 8:    256 dãy con,    0.07 ms
n=12:   4096 dãy con,    1.37 ms
n=16:  65536 dãy con,   26.25 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Đảo vai trò.** Thay vì sinh mọi dãy con của `s` rồi tra từ điển, hãy **lấy từng từ trong từ điển và hỏi: nó có là dãy con của `s` không?** Câu hỏi này chỉ cần **một lượt quét `s`** bằng hai con trỏ (`O(|s|)`), không cần sinh gì cả.

**Chọn đáp án bằng một thứ tự duy nhất.** "Dài nhất, hoà thì nhỏ nhất theo từ điển" chính là thứ tự so sánh theo khoá `(-len(w), w)`: từ nào có khoá **nhỏ nhất** là tốt nhất. Hai cách dùng khoá này:

1. Duyệt mọi từ, giữ lại từ hợp lệ có khoá nhỏ nhất (Cách 2).
2. **Sắp xếp** từ điển theo khoá rồi trả về từ hợp lệ **đầu tiên** gặp được, dừng ngay (Cách 3).

Hai hướng cải tiến nữa khi `s` dài và từ điển lớn: tiền xử lý `s` để mỗi từ kiểm tra theo `O(|từ|)` thay vì `O(|s|)` (Cách 4), hoặc quét `s` **một lần duy nhất** cho tất cả từ cùng lúc (Cách 5).

## 7. Các cách tiếp theo

Tổng cộng có 5 cách: Cách 1 (vét cạn dãy con, mục 5) và bốn cách dưới đây. Mỗi cách: giải thích từng đoạn → code demo → code nộp LeetCode → độ phức tạp.

### 7.1. Cách 2 — Hai con trỏ cho từng từ, giữ từ tốt nhất (`O(d · |s|)`)

Giải thích từng đoạn:

1. Hàm `check(t, s)`: con trỏ `i` trên `t`, quét `s`; khớp thì `i += 1`; kết quả `i == len(t)`.
2. `ans = ""` (chưa có đáp án).
3. Với mỗi từ `t` trong từ điển: nếu `check(t, s)` đúng **và** `t` "tốt hơn" `ans` thì cập nhật. "Tốt hơn" là `len(t) > len(ans)` hoặc (`len(t) == len(ans)` và `t < ans`).
4. Trả về `ans` (vẫn là `""` nếu không từ nào hợp lệ).

Lưu ý: khi `ans = ""` và `t` hợp lệ (độ dài ≥ 1) thì `len(t) > 0` đã đủ để thay.

In [12]:
def find_demo(s, dictionary):
    ans = ""
    for t in dictionary:
        ok = is_subseq(t, s)
        better = ok and (len(t) > len(ans) or (len(t) == len(ans) and t < ans))
        print(f"  t={t:<7} hợp lệ={ok!s:<5} tốt hơn ans={ans!r}? {better}")
        if better:
            ans = t
    return ans

print(find_demo("abpcplea", ["ale", "apple", "monkey", "plea"]))
print(find_demo("bab", ["ba", "ab", "a", "b"]))

  t=ale     hợp lệ=True  tốt hơn ans=''? True
  t=apple   hợp lệ=True  tốt hơn ans='ale'? True
  t=monkey  hợp lệ=False tốt hơn ans='apple'? False
  t=plea    hợp lệ=True  tốt hơn ans='apple'? False
apple
  t=ba      hợp lệ=True  tốt hơn ans=''? True
  t=ab      hợp lệ=True  tốt hơn ans='ba'? True
  t=a       hợp lệ=True  tốt hơn ans='ab'? False
  t=b       hợp lệ=True  tốt hơn ans='ab'? False
ab


In [13]:
class Sol2:
    """Cách 2: hai con trỏ cho từng từ, giữ từ tốt nhất."""
    def findLongestWord(self, s: str, dictionary: List[str]) -> str:
        def check(t: str, s: str) -> bool:
            i = 0
            for ch in s:
                if i < len(t) and ch == t[i]:
                    i += 1
            return i == len(t)

        ans = ""
        for t in dictionary:
            if check(t, s) and (len(t) > len(ans) or (len(t) == len(ans) and t < ans)):
                ans = t
        return ans

for s, dictionary, expected in examples:
    assert Sol2().findLongestWord(s, dictionary) == expected
print("Sol2 OK")

Sol2 OK


**Độ phức tạp:** gọi `d` là số từ, `m = |s|`, `L` là độ dài từ. Mỗi lần `check` quét tối đa `m` ký tự nên Time `O(d · m)` (và so sánh chuỗi `O(L)` mỗi lần cập nhật, tổng `O(d · L)`); với giới hạn đề ≈ `10^3 · 10^3 = 10^6` rất nhanh. Space `O(1)` (ngoài input).

### 7.2. Cách 3 — Sắp xếp theo `(-len, word)` rồi trả về từ hợp lệ đầu tiên

Ý tưởng: sắp xếp từ điển sao cho từ "tốt nhất" đứng trước. Khi đó **từ hợp lệ đầu tiên** trong thứ tự chính là đáp án, ta dừng ngay mà không cần so sánh độ dài/từ điển thêm.

1. `sorted(dictionary, key=lambda w: (-len(w), w))`.
2. Duyệt theo thứ tự đó; gặp `check(w, s)` đúng thì `return w`.
3. Không gặp: `return ""`.

Hay ở chỗ khi đáp án là từ dài, ta thường dừng sớm, và logic chọn đáp án đã "ẩn" vào `sort`.

In [14]:
def find_sorted_demo(s, dictionary):
    order = sorted(dictionary, key=lambda w: (-len(w), w))
    print("  thứ tự thử:", order)
    for w in order:
        ok = is_subseq(w, s)
        print(f"  thử {w:<7} -> {ok}")
        if ok:
            return w
    return ""

print(find_sorted_demo("abpcplea", ["ale", "apple", "monkey", "plea"]))
print(find_sorted_demo("abc", ["d", "cab"]))

  thứ tự thử: ['monkey', 'apple', 'plea', 'ale']
  thử monkey  -> False
  thử apple   -> True
apple
  thứ tự thử: ['cab', 'd']
  thử cab     -> False
  thử d       -> False



In [15]:
class Sol3:
    """Cách 3: sort theo (-len, word) rồi trả về từ hợp lệ đầu tiên."""
    def findLongestWord(self, s: str, dictionary: List[str]) -> str:
        def check(t: str) -> bool:
            i = 0
            for ch in s:
                if i < len(t) and ch == t[i]:
                    i += 1
            return i == len(t)

        for w in sorted(dictionary, key=lambda w: (-len(w), w)):
            if check(w):
                return w
        return ""

for s, dictionary, expected in examples:
    assert Sol3().findLongestWord(s, dictionary) == expected
print("Sol3 OK")

Sol3 OK


**Độ phức tạp:** sort `O(d log d · L)` (mỗi phép so sánh khoá tốn tới `O(L)`), rồi kiểm tra tối đa `d` từ, mỗi từ `O(m)`: tổng `O(d log d · L + d · m)` tệ nhất. Space `O(d)` cho list đã sắp xếp. Chậm hơn Cách 2 một chút trong tệ nhất nhưng **dừng sớm** nên thường nhanh; và rất ngắn gọn.

### 7.3. Cách 4 — Bảng "ký tự kế tiếp" (next-occurrence) để mỗi từ kiểm tra `O(|từ|)`

Cách 2 quét lại `s` cho mỗi từ. Nếu `s` dài còn từ ngắn/nhiều, ta tiền xử lý `s` một lần: `nxt[i][c]` = **chỉ số nhỏ nhất `>= i`** mà `s[idx] == c` (hoặc `n` nếu không có). Khi đó ghép một từ: `pos = 0`; với mỗi ký tự `ch` của từ: `p = nxt[pos][ch]`; nếu `p == n` thì thất bại, ngược lại `pos = p + 1`. Mỗi ký tự của từ chỉ tốn **một bước nhảy** `O(1)`.

Dựng bảng từ phải sang trái: `nxt[n]` toàn `n`; `nxt[i] = nxt[i+1]` rồi ghi `nxt[i][s[i]] = i`.

In [16]:
def build_next(s):
    n = len(s)
    nxt = [[n] * 26 for _ in range(n + 1)]
    for i in range(n - 1, -1, -1):
        nxt[i] = nxt[i + 1][:]
        nxt[i][ord(s[i]) - 97] = i
    return nxt

s = "abpcplea"
nxt = build_next(s)
print("i  :", "  ".join(f"{i}" for i in range(len(s) + 1)))
for c in "aplez":
    print(c, " :", "  ".join(f"{nxt[i][ord(c) - 97]}" for i in range(len(s) + 1)), "(8 = không còn)")

i  : 0  1  2  3  4  5  6  7  8
a  : 0  7  7  7  7  7  7  7  8 (8 = không còn)
p  : 2  2  2  4  4  8  8  8  8 (8 = không còn)
l  : 5  5  5  5  5  5  8  8  8 (8 = không còn)
e  : 6  6  6  6  6  6  6  8  8 (8 = không còn)
z  : 8  8  8  8  8  8  8  8  8 (8 = không còn)


In [17]:
def check_next(t, nxt, n):
    pos = 0
    for ch in t:
        p = nxt[pos][ord(ch) - 97]
        if p == n:
            return False
        pos = p + 1
    return True

print([check_next(t, nxt, len(s)) for t in ["ale", "apple", "monkey", "plea"]])

[True, True, False, True]


In [18]:
class Sol4:
    """Cách 4: bảng ký tự kế tiếp, mỗi từ kiểm tra O(len(từ))."""
    def findLongestWord(self, s: str, dictionary: List[str]) -> str:
        n = len(s)
        nxt = [[n] * 26 for _ in range(n + 1)]
        for i in range(n - 1, -1, -1):
            nxt[i] = nxt[i + 1][:]
            nxt[i][ord(s[i]) - 97] = i

        ans = ""
        for t in dictionary:
            pos, ok = 0, True
            for ch in t:
                p = nxt[pos][ord(ch) - 97]
                if p == n:
                    ok = False
                    break
                pos = p + 1
            if ok and (len(t) > len(ans) or (len(t) == len(ans) and t < ans)):
                ans = t
        return ans

for s, dictionary, expected in examples:
    assert Sol4().findLongestWord(s, dictionary) == expected
print("Sol4 OK")

Sol4 OK


**Độ phức tạp:** Time `O(26 · m + Σ|từ|)` (dựng bảng `O(26 m)` nhờ sao chép hàng, rồi tổng độ dài các từ). Space `O(26 · m)`. Hợp với bài toán "một `s`, rất nhiều truy vấn từ" (xem LeetCode 392 follow-up, mục 15). Với ràng buộc nhỏ của bài này, bảng nặng hơn lợi ích, nhưng ý tưởng "nhảy cóc" rất đáng học.

### 7.4. Cách 5 — Quét `s` một lần với các "xô chờ"

Ý tưởng khác hẳn: thay vì xử lý từng từ trọn vẹn, ta **quét `s` đúng một lần** và cho **tất cả từ cùng tiến lên**. Với mỗi từ ta nhớ nó đang **chờ ký tự nào** (ký tự tiếp theo cần ghép). Gom các từ theo ký tự đang chờ vào "xô" `waiting[c]`.

1. Ban đầu mỗi từ ở vị trí `0`, nằm trong xô `waiting[w[0]]`.
2. Với mỗi `ch` trong `s`: lấy **toàn bộ xô** `waiting[ch]` ra (và xoá xô); mỗi từ trong xô tiến thêm 1 vị trí.
3. Từ nào đã ghép hết → bỏ vào danh sách `done`; từ chưa hết → đưa vào xô của ký tự mới mà nó chờ.
4. Cuối cùng chọn từ tốt nhất trong `done` bằng khoá `(-len, từ)`.

Vì lấy hẳn xô ra rồi mới thêm vào xô mới, một từ cần hai ký tự giống nhau liên tiếp (như `pp`) sẽ phải đợi tới ký tự `s` **kế tiếp** — đúng ngữ nghĩa dãy con.

In [19]:
def buckets_demo(s, dictionary):
    waiting = defaultdict(list)
    for w in dictionary:
        waiting[w[0]].append((w, 0))
    done = []
    for j, ch in enumerate(s):
        bucket = waiting.pop(ch, [])
        if bucket:
            print(f"  j={j} ch={ch}: xô = {[w for w, _ in bucket]}")
        for w, p in bucket:
            p += 1
            if p == len(w):
                done.append(w)
            else:
                waiting[w[p]].append((w, p))
    return done

done = buckets_demo("abpcplea", ["ale", "apple", "monkey", "plea"])
print("done =", done)

  j=0 ch=a: xô = ['ale', 'apple']
  j=2 ch=p: xô = ['plea', 'apple']
  j=4 ch=p: xô = ['apple']
  j=5 ch=l: xô = ['ale', 'plea', 'apple']
  j=6 ch=e: xô = ['ale', 'plea', 'apple']
  j=7 ch=a: xô = ['plea']
done = ['ale', 'apple', 'plea']


In [20]:
class Sol5:
    """Cách 5: quét s một lần, các từ chờ ký tự trong từng xô."""
    def findLongestWord(self, s: str, dictionary: List[str]) -> str:
        waiting = defaultdict(list)
        for w in dictionary:
            waiting[w[0]].append((w, 0))
        ans = ""
        for ch in s:
            bucket = waiting.pop(ch, [])
            for w, p in bucket:
                p += 1
                if p == len(w):
                    if len(w) > len(ans) or (len(w) == len(ans) and w < ans):
                        ans = w
                else:
                    waiting[w[p]].append((w, p))
        return ans

for s, dictionary, expected in examples:
    assert Sol5().findLongestWord(s, dictionary) == expected
print("Sol5 OK")

Sol5 OK


**Độ phức tạp:** Time `O(m + Σ|từ|)`: mỗi từ chỉ tiến tối đa `|từ|` bước, mỗi bước `O(1)`, cộng với một lượt quét `s`. Space `O(d)` cho các xô. Đây là kỹ thuật của LeetCode 792 (Number of Matching Subsequences) và nhanh nhất về tiệm cận, dù code dài hơn.

## 8. TRACE từng bước

### 8.1. Trace hai con trỏ cho một từ

Bảng trạng thái: `j` là chỉ số trên `s`, `i` là số ký tự của `t` đã ghép.

| `j` | `s[j]` | `t[i]` (cần ghép) | khớp? | `i` sau |
|---|---|---|---|---|
| 0 | a | a | có | 1 |
| 1 | b | p | không | 1 |
| 2 | p | p | có | 2 |
| 3 | c | p | không | 2 |
| 4 | p | p | có | 3 |
| 5 | l | l | có | 4 |
| 6 | e | e | có | 5 |
| 7 | a | — (hết) | — | 5 |

(với `s = "abpcplea"`, `t = "apple"`). Ta in bảng này bằng code:

In [21]:
def trace_two_pointers(t, s):
    i = 0
    print(f"t = {t!r}, s = {s!r}")
    for j, ch in enumerate(s):
        need = t[i] if i < len(t) else "—"
        hit = i < len(t) and ch == t[i]
        if hit:
            i += 1
        print(f"  j={j} s[j]={ch} cần={need} khớp={'có' if hit else 'không':<5} -> i={i}")
    print("  kết quả:", i == len(t))
    return i == len(t)

trace_two_pointers("apple", "abpcplea")
trace_two_pointers("monkey", "abpcplea")

t = 'apple', s = 'abpcplea'
  j=0 s[j]=a cần=a khớp=có    -> i=1
  j=1 s[j]=b cần=p khớp=không -> i=1
  j=2 s[j]=p cần=p khớp=có    -> i=2
  j=3 s[j]=c cần=p khớp=không -> i=2
  j=4 s[j]=p cần=p khớp=có    -> i=3
  j=5 s[j]=l cần=l khớp=có    -> i=4
  j=6 s[j]=e cần=e khớp=có    -> i=5
  j=7 s[j]=a cần=— khớp=không -> i=5
  kết quả: True
t = 'monkey', s = 'abpcplea'
  j=0 s[j]=a cần=m khớp=không -> i=0
  j=1 s[j]=b cần=m khớp=không -> i=0
  j=2 s[j]=p cần=m khớp=không -> i=0
  j=3 s[j]=c cần=m khớp=không -> i=0
  j=4 s[j]=p cần=m khớp=không -> i=0
  j=5 s[j]=l cần=m khớp=không -> i=0
  j=6 s[j]=e cần=m khớp=không -> i=0
  j=7 s[j]=a cần=m khớp=không -> i=0
  kết quả: False


False

Quan sát: với `monkey`, ký tự `m` không bao giờ khớp nên `i` đứng yên ở `0` suốt lượt quét → `False`.

### 8.2. Trace vòng chọn đáp án (Cách 2) với luật hoà

`s = "bab"`, từ điển `["ba", "ab", "a", "b"]`. Ta in `ans` sau mỗi từ:

In [22]:
def trace_select(s, dictionary):
    ans = ""
    for t in dictionary:
        ok = is_subseq(t, s)
        if not ok:
            reason = "không là dãy con"
        elif len(t) > len(ans):
            reason = f"dài hơn {ans!r} ({len(t)} > {len(ans)}) -> nhận"
            ans = t
        elif len(t) == len(ans) and t < ans:
            reason = f"cùng độ dài, {t!r} < {ans!r} -> nhận"
            ans = t
        else:
            reason = "không tốt hơn ans"
        print(f"t={t!r:<8} {reason:<45} ans={ans!r}")
    return ans

print("kết quả:", trace_select("bab", ["ba", "ab", "a", "b"]))
print("kết quả:", trace_select("abpcplea", ["ale", "apple", "monkey", "plea"]))

t='ba'     dài hơn '' (2 > 0) -> nhận                    ans='ba'
t='ab'     cùng độ dài, 'ab' < 'ba' -> nhận              ans='ab'
t='a'      không tốt hơn ans                             ans='ab'
t='b'      không tốt hơn ans                             ans='ab'
kết quả: ab
t='ale'    dài hơn '' (3 > 0) -> nhận                    ans='ale'
t='apple'  dài hơn 'ale' (5 > 3) -> nhận                 ans='apple'
t='monkey' không là dãy con                              ans='apple'
t='plea'   không tốt hơn ans                             ans='apple'
kết quả: apple


Ở `"bab"`: `"ba"` nhận trước (dài hơn `""`); `"ab"` cùng độ dài 2 nhưng nhỏ hơn nên thay; `"a"`, `"b"` ngắn hơn bị loại.

### 8.3. Trace Cách 3 (sort) — thứ tự thử và dừng sớm

Ta in danh sách khoá `(-len, từ)` để thấy vì sao thứ tự sort chính là thứ tự ưu tiên.

In [23]:
d = ["ale", "apple", "monkey", "plea", "pleb", "applf"]
for w in sorted(d, key=lambda w: (-len(w), w)):
    print((-len(w), w), "| dãy con của 'abpcplea'?", is_subseq(w, "abpcplea"))
print("-> từ hợp lệ đầu tiên theo thứ tự trên là đáp án:", Sol3().findLongestWord("abpcplea", d))

(-6, 'monkey') | dãy con của 'abpcplea'? False
(-5, 'apple') | dãy con của 'abpcplea'? True
(-5, 'applf') | dãy con của 'abpcplea'? False
(-4, 'plea') | dãy con của 'abpcplea'? True
(-4, 'pleb') | dãy con của 'abpcplea'? False
(-3, 'ale') | dãy con của 'abpcplea'? True
-> từ hợp lệ đầu tiên theo thứ tự trên là đáp án: apple


### 8.4. Trace Cách 5 (xô chờ)

Mỗi bước in nội dung các xô để thấy các từ "nhích" qua từng ký tự của `s`.

In [24]:
def trace_buckets(s, dictionary):
    waiting = defaultdict(list)
    for w in dictionary:
        waiting[w[0]].append((w, 0))
    show = lambda: {c: [f"{w}@{p}" for w, p in v] for c, v in sorted(waiting.items())}
    print("ban đầu:", show())
    done = []
    for j, ch in enumerate(s):
        bucket = waiting.pop(ch, [])
        for w, p in bucket:
            p += 1
            if p == len(w):
                done.append(w)
            else:
                waiting[w[p]].append((w, p))
        print(f"j={j} ch={ch} xử lý {len(bucket)} từ | done={done} | xô={show()}")
    return done

trace_buckets("abpcplea", ["ale", "apple", "monkey", "plea"])

ban đầu: {'a': ['ale@0', 'apple@0'], 'm': ['monkey@0'], 'p': ['plea@0']}
j=0 ch=a xử lý 2 từ | done=[] | xô={'l': ['ale@1'], 'm': ['monkey@0'], 'p': ['plea@0', 'apple@1']}
j=1 ch=b xử lý 0 từ | done=[] | xô={'l': ['ale@1'], 'm': ['monkey@0'], 'p': ['plea@0', 'apple@1']}
j=2 ch=p xử lý 2 từ | done=[] | xô={'l': ['ale@1', 'plea@1'], 'm': ['monkey@0'], 'p': ['apple@2']}
j=3 ch=c xử lý 0 từ | done=[] | xô={'l': ['ale@1', 'plea@1'], 'm': ['monkey@0'], 'p': ['apple@2']}
j=4 ch=p xử lý 1 từ | done=[] | xô={'l': ['ale@1', 'plea@1', 'apple@3'], 'm': ['monkey@0']}
j=5 ch=l xử lý 3 từ | done=[] | xô={'e': ['ale@2', 'plea@2', 'apple@4'], 'm': ['monkey@0']}
j=6 ch=e xử lý 3 từ | done=['ale', 'apple'] | xô={'a': ['plea@3'], 'm': ['monkey@0']}
j=7 ch=a xử lý 1 từ | done=['ale', 'apple', 'plea'] | xô={'m': ['monkey@0']}


['ale', 'apple', 'plea']

Ở `j=2` (`p`): xô `p` chứa `plea@0` và... `apple@1` (vì `apple` đã ghép `a`, đang chờ `p`). Cả hai cùng tiến. Ở `j=4` (`p` lần hai): `apple@2` đang chờ `p` thứ hai và tiến tiếp; `monkey` chờ `m` mãi mãi nên nằm im.

## 9. Code nộp LeetCode

Dán đoạn dưới vào LeetCode (trên LeetCode đã có sẵn `List`).

In [25]:
class Solution:
    def findLongestWord(self, s: str, dictionary: List[str]) -> str:
        def check(t: str, s: str) -> bool:
            # t có phải dãy con của s không? (hai con trỏ)
            i = 0
            for ch in s:
                if i < len(t) and ch == t[i]:
                    i += 1
            return i == len(t)

        ans = ""
        for t in dictionary:
            if check(t, s) and (len(t) > len(ans) or (len(t) == len(ans) and t < ans)):
                ans = t
        return ans


for s, dictionary, expected in examples:
    assert Solution().findLongestWord(s, dictionary) == expected
print("Solution chạy đúng 5 ví dụ")

Solution chạy đúng 5 ví dụ


## 10. Vì sao thuật toán đúng

Hai phần: (A) kiểm tra dãy con bằng hai con trỏ đúng; (B) cách chọn đáp án đúng.

### (A) Ghép tham lam là đủ

**Mệnh đề.** `t` là dãy con của `s` khi và chỉ khi thuật toán hai con trỏ kết thúc với `i == len(t)`.

*Chứng minh.* (⇐) Mỗi lần `i` tăng, ta ghép `t[i]` với một ký tự `s[j]` ở vị trí **lớn hơn** vị trí đã ghép trước đó (vì `j` chỉ tăng). Ghép đủ `len(t)` ký tự theo đúng thứ tự → `t` là dãy con.

(⇒) Giả sử `t` là dãy con bằng một cách ghép bất kỳ `j_0 < j_1 < ... < j_{r-1}`. Gọi `g_k` là vị trí mà thuật toán tham lam ghép `t[k]` — ta chứng minh quy nạp `g_k <= j_k`. Với `k = 0`: `g_0` là vị trí đầu tiên mà `s` có `t[0]`, nên `g_0 <= j_0`. Giả sử `g_{k-1} <= j_{k-1} < j_k`; tham lam tìm `t[k]` ở vị trí **đầu tiên sau `g_{k-1}`**; vì `j_k > g_{k-1}` là một vị trí hợp lệ chứa `t[k]`, nên `g_k <= j_k`. Do đó tham lam không bao giờ bị "kẹt" nếu có cách ghép nào đó. ∎

Ý nghĩa: **ghép sớm nhất luôn không tệ hơn** bất kỳ cách ghép nào khác — còn lại nhiều ký tự phía sau nhất.

### (B) Chọn đáp án

Định nghĩa thứ tự "tốt hơn" `A ≺ B` khi `len(A) > len(B)`, hoặc `len(A) == len(B)` và `A < B` (từ điển). Đây là một **thứ tự toàn phần chặt** trên các chuỗi (hai chuỗi khác nhau luôn so sánh được; hai chuỗi bằng nhau thì không "tốt hơn" nhau). Thuật toán Cách 2 duy trì bất biến: *sau khi xử lý các từ đầu tiên, `ans` là phần tử `≺`-nhỏ nhất trong các từ hợp lệ đã thấy (hoặc `""` nếu chưa có)*. Khi gặp `t` hợp lệ và `t ≺ ans` ta thay `ans = t`; trường hợp `ans = ""` thì mọi từ hợp lệ (độ dài ≥ 1) đều `≺ ""`. Cuối cùng `ans` là phần tử tốt nhất trong mọi từ hợp lệ — đúng yêu cầu đề. Từ lặp lại không ảnh hưởng vì `t ≺ ans` sai khi `t == ans`.

Cách 3 đúng vì sort theo khoá `(-len, w)` cho **đúng** thứ tự `≺`, nên từ hợp lệ đầu tiên là phần tử `≺`-nhỏ nhất của tập từ hợp lệ.

Cách 4 đúng vì `nxt[pos][c]` chính là "vị trí đầu tiên `>= pos` chứa `c`" — chính là bước tham lam ở (A). Cách 5 đúng vì mỗi từ chỉ tiến khi gặp **đúng ký tự nó đang chờ** ở vị trí `s` sau vị trí đã ghép trước (xô được lấy ra rồi mới đưa vào xô mới), tức cũng là ghép tham lam.

## 11. Phân tích Time / Space Complexity

Ký hiệu: `m = |s|`, `d` = số từ, `L` = độ dài từ trung bình (`Σ|từ| = d · L`).

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê dãy con của `s` | `O(2^m · m)` | `O(2^m · m)` |
| 2. Hai con trỏ từng từ | `O(d · m)` (+ `O(d · L)` so sánh chuỗi) | `O(1)` |
| 3. Sort `(-len, w)` + thử | `O(d log d · L + d · m)` | `O(d)` |
| 4. Bảng ký tự kế tiếp | `O(26 m + d · L)` | `O(26 m)` |
| 5. Quét một lần, xô chờ | `O(m + d · L)` | `O(d)` |

Với `m = d = L = 1000`: Cách 2 ≈ `10^6` phép — chạy tức thì. Ta đo thực nghiệm trên dữ liệu lớn nhất đề cho phép (trường hợp khó: nhiều từ **hợp lệ và dài** để Cách 2 phải quét gần hết `s`).

In [26]:
rng = random.Random(524)
S = "".join(rng.choice("abc") for _ in range(1000))

def random_subseq(s, rng, keep):
    return "".join(ch for ch in s if rng.random() < keep)

big_dict = [random_subseq(S, rng, rng.choice([0.2, 0.5, 0.9])) or "a" for _ in range(500)]
big_dict += ["".join(rng.choice("abc") for _ in range(rng.randint(1, 1000))) for _ in range(500)]
assert len(big_dict) == 1000 and all(1 <= len(w) <= 1000 for w in big_dict)

results = {}
for name, cls in [("Sol2", Sol2), ("Sol3", Sol3), ("Sol4", Sol4), ("Sol5", Sol5), ("Solution", Solution)]:
    t0 = time.perf_counter()
    res = cls().findLongestWord(S, big_dict)
    results[name] = res
    print(f"{name:<9}: {1000 * (time.perf_counter() - t0):8.2f} ms -> độ dài đáp án {len(res)}")
assert len(set(results.values())) == 1
print("cả 5 cách cho cùng đáp án trên dữ liệu lớn")

Sol2     :    85.24 ms -> độ dài đáp án 922
Sol3     :     4.65 ms -> độ dài đáp án 922
Sol4     :    33.32 ms -> độ dài đáp án 922
Sol5     :    75.21 ms -> độ dài đáp án 922
Solution :    75.11 ms -> độ dài đáp án 922
cả 5 cách cho cùng đáp án trên dữ liệu lớn


## 12. Bộ test

### 12.1. Ví dụ đề và edge case

**Oracle độc lập:** vét cạn trên **mọi dãy con của `s`** (Cách 1) — không dùng hai con trỏ, bảng hay xô. Ta gom các cách vào dict `solvers`.

In [27]:
def oracle(s, dictionary):
    subs = all_subsequences(s)
    valid = [w for w in dictionary if w in subs]
    return min(valid, key=lambda w: (-len(w), w)) if valid else ""

solvers = {
    "brute": brute_force,
    "two_pointers": lambda s, d: Sol2().findLongestWord(s, d),
    "sorted": lambda s, d: Sol3().findLongestWord(s, d),
    "next_table": lambda s, d: Sol4().findLongestWord(s, d),
    "buckets": lambda s, d: Sol5().findLongestWord(s, d),
    "leetcode": lambda s, d: Solution().findLongestWord(s, d),
}

for s, dictionary, expected in examples:
    assert oracle(s, dictionary) == expected
    for name, fn in solvers.items():
        assert fn(s, dictionary) == expected, (name, s, dictionary)
print("Ví dụ đề: OK cho cả 6 solver và oracle")

Ví dụ đề: OK cho cả 6 solver và oracle


In [28]:
edge_cases = [
    ("a", ["a"], "a", "s một ký tự, trùng từ"),
    ("a", ["b"], "", "không có ký tự chung"),
    ("a", ["aa"], "", "từ dài hơn s"),
    ("abc", ["abc"], "abc", "từ đúng bằng s (xoá 0 ký tự)"),
    ("abc", ["abcd"], "", "từ dài hơn s một ký tự"),
    ("abc", ["cab"], "", "đủ ký tự nhưng sai thứ tự"),
    ("abc", ["ca", "cb", "ba"], "", "mọi từ sai thứ tự"),
    ("abc", ["ac", "ab", "bc"], "ab", "ba từ cùng độ dài, lấy nhỏ nhất"),
    ("abc", ["bc", "ac", "ab"], "ab", "thứ tự trong từ điển không ảnh hưởng"),
    ("aaa", ["a", "aa", "aaa", "aaaa"], "aaa", "ký tự lặp lại"),
    ("bab", ["ba", "ab", "a", "b"], "ab", "luật hoà từ điển"),
    ("abpcplea", ["ale", "apple", "monkey", "plea"], "apple", "ví dụ 1"),
    ("abpcplea", ["a", "b", "c"], "a", "ví dụ 2"),
    ("abpcplea", ["plea", "ale", "applf", "applb"], "plea", "applf/applb không là dãy con"),
    ("abpcplea", ["pleb", "plea"], "plea", "chỉ một từ hợp lệ"),
    ("abc", ["abc", "abc", "ab"], "abc", "từ lặp lại"),
    ("zyx", ["z", "y", "x", "zy", "yx", "zx", "xyz"], "yx", "ba từ độ dài 2 hợp lệ: zy, yx, zx -> nhỏ nhất yx"),
    ("ab" * 5, ["aaaaa", "bbbbb", "ababababab", "bbbbbb"], "ababababab", "s = ababababab"),
    ("abcde", ["ae", "ad", "abcde", "abcdf"], "abcde", "chọn từ dài nhất"),
]
for s, dictionary, expected, note in edge_cases:
    assert oracle(s, dictionary) == expected, ("oracle", note, oracle(s, dictionary))
    for name, fn in solvers.items():
        assert fn(s, dictionary) == expected, (name, note)
print(f"{len(edge_cases)} edge case: OK")

19 edge case: OK


### 12.2. Phản ví dụ cho các ý tưởng sai

Những "ý tưởng nghe hợp lý" nhưng sai (các lỗi chi tiết ở mục 13). Ta kiểm tra chúng thật sự sai trên những đầu vào cụ thể:

- Dùng `t in s` (chuỗi con): sai với `s = "abpcplea"`, `t = "apple"`.
- Chỉ so độ dài, không có luật hoà: sai với `s = "bab"`, `dictionary = ["ba","ab"]`.
- Chỉ kiểm tra "đủ ký tự" (không xét thứ tự): sai với `s = "abc"`, `dictionary = ["cab"]`.

In [29]:
def wrong_substring(s, dictionary):
    ans = ""
    for t in dictionary:
        if t in s and (len(t) > len(ans) or (len(t) == len(ans) and t < ans)):
            ans = t
    return ans

def wrong_no_tiebreak(s, dictionary):
    ans = ""
    for t in dictionary:
        if is_subseq(t, s) and len(t) > len(ans):
            ans = t
    return ans

def wrong_letters_only(s, dictionary):
    ans = ""
    for t in dictionary:
        if all(t.count(c) <= s.count(c) for c in set(t)) and (len(t) > len(ans) or (len(t) == len(ans) and t < ans)):
            ans = t
    return ans

cases = [("abpcplea", ["ale", "apple", "monkey", "plea"]), ("bab", ["ba", "ab"]), ("abc", ["cab"])]
for s, d in cases:
    print(f"s={s!r} d={d} | đúng={oracle(s, d)!r} | chuỗi con={wrong_substring(s, d)!r} | không hoà={wrong_no_tiebreak(s, d)!r} | đủ chữ={wrong_letters_only(s, d)!r}")
assert wrong_substring(*cases[0]) != oracle(*cases[0])
assert wrong_no_tiebreak(*cases[1]) != oracle(*cases[1])
assert wrong_letters_only(*cases[2]) != oracle(*cases[2])

s='abpcplea' d=['ale', 'apple', 'monkey', 'plea'] | đúng='apple' | chuỗi con='plea' | không hoà='apple' | đủ chữ='apple'
s='bab' d=['ba', 'ab'] | đúng='ab' | chuỗi con='ab' | không hoà='ba' | đủ chữ='ab'
s='abc' d=['cab'] | đúng='' | chuỗi con='' | không hoà='' | đủ chữ='cab'


### 12.3. Đối chiếu mọi cách bằng vét cạn toàn bộ input nhỏ

Duyệt **mọi** `s` trên bảng chữ cái `{a, b}` có độ dài `1..6`, và với mỗi `s` thử: từ điển gồm **mọi từ độ dài 1..3** trên `{a, b}`, cùng nhiều tập con ngẫu nhiên (có seed) của nó (kể cả có từ lặp).

In [30]:
from itertools import product

all_words = ["".join(p) for n in range(1, 4) for p in product("ab", repeat=n)]
print("số từ:", len(all_words))
rng = random.Random(524)
checked = 0
for n in range(1, 7):
    for tup in product("ab", repeat=n):
        s = "".join(tup)
        dicts = [all_words]
        for _ in range(8):
            dicts.append([rng.choice(all_words) for _ in range(rng.randint(1, 6))])
        for dictionary in dicts:
            expected = oracle(s, dictionary)
            for name, fn in solvers.items():
                assert fn(s, dictionary) == expected, (name, s, dictionary)
            checked += 1
print("đã kiểm", checked, "cặp (s, dictionary): tất cả khớp oracle")

số từ: 14
đã kiểm 1134 cặp (s, dictionary): tất cả khớp oracle


### 12.4. Test ngẫu nhiên có seed (đúng ràng buộc)

Sinh `s` độ dài `1..12` trên bảng chữ cái nhỏ (`a`–`c`) để nhiều từ hợp lệ và nhiều trường hợp hoà; từ điển `1..10` từ có độ dài `1..8`, một phần được cố ý lấy từ dãy con của `s`. Oracle vét cạn dùng `s` ngắn nên vẫn nhanh.

In [31]:
rng = random.Random(5240)

def gen(rng):
    n = rng.randint(1, 12)
    alpha = rng.choice(["ab", "abc", "abcd"])
    s = "".join(rng.choice(alpha) for _ in range(n))
    d = []
    for _ in range(rng.randint(1, 10)):
        if rng.random() < 0.5:
            w = "".join(ch for ch in s if rng.random() < 0.6) or rng.choice(alpha)
        else:
            w = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 8)))
        d.append(w)
    assert 1 <= len(s) <= 1000 and 1 <= len(d) <= 1000 and all(1 <= len(w) <= 1000 for w in d)
    return s, d

nonempty = 0
for _ in range(4000):
    s, dictionary = gen(rng)
    expected = oracle(s, dictionary)
    nonempty += expected != ""
    for name, fn in solvers.items():
        assert fn(s, dictionary) == expected, (name, s, dictionary)
print("4000 test ngẫu nhiên: OK;", nonempty, "ca có đáp án,", 4000 - nonempty, "ca trả về rỗng")

4000 test ngẫu nhiên: OK; 3737 ca có đáp án, 263 ca trả về rỗng


### 12.5. Test lớn và kiểm tra tính hợp lệ của kết quả

Với `|s| = 1000` ta không thể dùng oracle vét cạn; thay vào đó (a) so 4 cách đa thức với nhau và (b) kiểm tra **tính chất**: đáp án (nếu khác rỗng) nằm trong từ điển, là dãy con của `s`, và **không có từ hợp lệ nào tốt hơn** (kiểm tra độc lập bằng `iter`-trick).

In [32]:
def valid_answer(s, dictionary, ans):
    it_ok = lambda t: (lambda it: all(c in it for c in t))(iter(s))
    good = [w for w in dictionary if it_ok(w)]
    if not good:
        return ans == ""
    best = min(good, key=lambda w: (-len(w), w))
    return ans in dictionary and it_ok(ans) and ans == best

rng = random.Random(52400)
fast = ["two_pointers", "sorted", "next_table", "buckets", "leetcode"]
for trial in range(15):
    m = rng.choice([50, 300, 1000])
    alpha = rng.choice(["ab", "abc", "abcdefghijklmnopqrstuvwxyz"])
    s = "".join(rng.choice(alpha) for _ in range(m))
    d = []
    for _ in range(rng.choice([10, 200, 1000])):
        if rng.random() < 0.5:
            w = "".join(ch for ch in s if rng.random() < rng.choice([0.1, 0.5, 0.9])) or "a"
        else:
            w = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 1000)))
        d.append(w[:1000])
    answers = {solvers[name](s, d) for name in fast}
    assert len(answers) == 1, trial
    assert valid_answer(s, d, answers.pop()), trial
print("15 test lớn: 5 solver nhất quán và đáp án hợp lệ")

# trường hợp tệ nhất cho cách 2: s toàn 'a', từ toàn 'a' độ dài 1000 (mọi lần check quét hết s)
s = "a" * 1000
d = ["a" * 1000 for _ in range(1000)]
assert all(solvers[name](s, d) == "a" * 1000 for name in fast)
d = ["a" * 999 + "b" for _ in range(1000)]
assert all(solvers[name](s, d) == "" for name in fast)
print("trường hợp tệ nhất: OK")

15 test lớn: 5 solver nhất quán và đáp án hợp lệ


trường hợp tệ nhất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `in` (chuỗi con) thay vì dãy con

**Code sai:**

In [33]:
class WrongSolution1:
    def findLongestWord(self, s, dictionary):
        ans = ""
        for t in dictionary:
            if t in s and (len(t) > len(ans) or (len(t) == len(ans) and t < ans)):   # sai: in = chuỗi con
                ans = t
        return ans

s, d = "abpcplea", ["ale", "apple", "monkey", "plea"]
print("sai  :", WrongSolution1().findLongestWord(s, d), "(đúng phải là 'apple')")
print("đúng :", Solution().findLongestWord(s, d))

sai  : plea (đúng phải là 'apple')
đúng : apple


**Vì sao sai:** `"apple" in "abpcplea"` là `False` vì `apple` không **liền nhau** trong `s`, trong khi nó là dãy con (xoá `b`, `c`, `a`). **Code đúng:** dùng hai con trỏ (`check`) thay cho `in`.

### Lỗi 2: Không xử lý luật hoà theo từ điển

In [34]:
class WrongSolution2:
    def findLongestWord(self, s, dictionary):
        ans = ""
        for t in dictionary:
            if is_subseq(t, s) and len(t) > len(ans):      # chỉ so độ dài: từ đầu tiên thắng
                ans = t
        return ans

for s, d in [("bab", ["ba", "ab"]), ("abpcplea", ["b", "a", "c"])]:
    print(d, "| sai:", repr(WrongSolution2().findLongestWord(s, d)), "| đúng:", repr(Solution().findLongestWord(s, d)))

['ba', 'ab'] | sai: 'ba' | đúng: 'ab'
['b', 'a', 'c'] | sai: 'b' | đúng: 'a'


**Vì sao sai:** với `["ba", "ab"]` cả hai cùng độ dài 2, ta giữ lại `"ba"` (đến trước) thay vì `"ab"` (nhỏ hơn theo từ điển). Kết quả phụ thuộc **thứ tự** trong từ điển — dấu hiệu sai. **Code đúng:** thêm nhánh `len(t) == len(ans) and t < ans`.

### Lỗi 3: Chỉ kiểm tra "đủ chữ cái", bỏ qua thứ tự

In [35]:
from collections import Counter

class WrongSolution3:
    def findLongestWord(self, s, dictionary):
        have = Counter(s)
        ans = ""
        for t in dictionary:
            need = Counter(t)
            if all(have[c] >= v for c, v in need.items()) and (len(t) > len(ans) or (len(t) == len(ans) and t < ans)):
                ans = t
        return ans

print("sai  :", repr(WrongSolution3().findLongestWord("abc", ["d", "cab"])), "(đúng phải là '')")
print("đúng :", repr(Solution().findLongestWord("abc", ["d", "cab"])))

sai  : 'cab' (đúng phải là '')
đúng : ''


**Vì sao sai:** `"cab"` dùng đúng các chữ `a, b, c` của `"abc"` nhưng **sai thứ tự** (`c` đứng trước `a`) nên không thể tạo bằng cách xoá. Dãy con yêu cầu **giữ nguyên thứ tự**. **Code đúng:** hai con trỏ.

### Lỗi 4: `IndexError` vì con trỏ chạy quá cuối từ

In [36]:
def wrong_check(t, s):
    i = 0
    for ch in s:
        if ch == t[i]:           # sai: khi i == len(t) thì t[i] vượt chỉ số
            i += 1
    return i == len(t)

try:
    print(wrong_check("a", "abpcplea"))
except IndexError as e:
    print("lỗi:", type(e).__name__, "-", e)
print("đúng:", is_subseq("a", "abpcplea"))

lỗi: IndexError - string index out of range
đúng: True


**Vì sao sai:** sau khi ghép đủ ký tự, `i == len(t)` và `t[i]` không tồn tại. **Code đúng:** kiểm tra `i < len(t)` trước (như `is_subseq`), hoặc `break` khi `i == len(t)`.

### Lỗi 5: Dùng `max(..., key=len)` hoặc `max` theo `(len, w)`

In [37]:
def wrong_max_len(s, dictionary):
    valid = [t for t in dictionary if is_subseq(t, s)]
    return max(valid, key=len) if valid else ""

def wrong_max_tuple(s, dictionary):
    valid = [t for t in dictionary if is_subseq(t, s)]
    return max(valid, key=lambda w: (len(w), w)) if valid else ""    # lấy từ LỚN nhất theo từ điển

for s, d in [("bab", ["ba", "ab"]), ("bab", ["ab", "ba"])]:
    print(d, "| max(key=len):", wrong_max_len(s, d), "| max (len, w):", wrong_max_tuple(s, d), "| đúng:", Solution().findLongestWord(s, d))

['ba', 'ab'] | max(key=len): ba | max (len, w): ba | đúng: ab
['ab', 'ba'] | max(key=len): ab | max (len, w): ba | đúng: ab


**Vì sao sai:** `max(..., key=len)` trả về phần tử **đầu tiên** đạt max (phụ thuộc thứ tự input); `max` theo `(len, w)` lại chọn **từ điển lớn nhất** — ngược với đề. **Code đúng:** `min(valid, key=lambda w: (-len(w), w))`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **"xoá một số ký tự" / "giữ thứ tự"** → đó là **dãy con (subsequence)**, nghĩ ngay hai con trỏ.
- Nhiều tiêu chí xếp hạng ("dài nhất, hoà thì từ điển nhỏ nhất") → gộp thành **một khoá tuple** `(-len, w)` và dùng `min`/`sorted`; tránh viết `if` lồng nhau dễ sót.
- Một chuỗi `s` cố định, rất nhiều truy vấn "có là dãy con không" → tiền xử lý (bảng ký tự kế tiếp, hoặc danh sách vị trí + `bisect`) hoặc gom các truy vấn và quét `s` một lần (xô chờ).
- Phân biệt: `substring` (liền nhau, dùng `in`/`find`), `subsequence` (giữ thứ tự, hai con trỏ), `anagram/đủ chữ` (đếm, `Counter`).
- Muốn dừng sớm: sắp xếp theo thứ tự ưu tiên rồi `return` phần tử hợp lệ đầu tiên.
- Luôn thử tay: từ dài hơn `s`, từ trùng hoàn toàn `s`, ký tự lặp (`"aaa"`), từ điển có phần tử lặp, không có đáp án.
- Bảng chữ cái chỉ 26 chữ thường → mảng 26 phần tử thay cho `dict` (nhanh và gọn).

## 15. Biến thể

### 15.1. Trả về mọi từ dài nhất (có thể nhiều từ)

Thay vì chỉ lấy từ nhỏ nhất theo từ điển, trả về **danh sách các từ khác nhau có độ dài lớn nhất**, sắp xếp từ điển.

In [38]:
def all_longest(s, dictionary):
    valid = {w for w in dictionary if is_subseq(w, s)}
    if not valid:
        return []
    L = max(map(len, valid))
    return sorted(w for w in valid if len(w) == L)

def all_longest_oracle(s, dictionary):
    subs = all_subsequences(s)
    valid = {w for w in dictionary if w in subs}
    if not valid:
        return []
    L = max(len(w) for w in valid)
    return sorted(w for w in valid if len(w) == L)

print(all_longest("abc", ["ab", "bc", "ac", "abc", "d"]), all_longest("abc", ["ab", "bc", "ac", "x"]))
rng = random.Random(151)
for _ in range(1500):
    s, d = gen(rng)
    assert all_longest(s, d) == all_longest_oracle(s, d)
print("all_longest khớp oracle")

['abc'] ['ab', 'ac', 'bc']
all_longest khớp oracle


### 15.2. LeetCode 392 và follow-up: một `s`, rất nhiều truy vấn "có là dãy con không"

Ta lập `pos[c]` = danh sách (đã sắp xếp) các chỉ số mà `s` có chữ `c`. Với mỗi truy vấn `t`, giữ `cur` (chỉ số đã ghép cuối) rồi dùng `bisect_right` tìm vị trí **đầu tiên `> cur`** của chữ tiếp theo — `O(|t| log |s|)` mỗi truy vấn, không quét `s`.

In [39]:
class SubseqQuery:
    def __init__(self, s):
        self.pos = defaultdict(list)
        for i, ch in enumerate(s):
            self.pos[ch].append(i)

    def is_subsequence(self, t):
        cur = -1
        for ch in t:
            lst = self.pos.get(ch)
            if not lst:
                return False
            k = bisect_right(lst, cur)          # phần tử đầu tiên > cur
            if k == len(lst):
                return False
            cur = lst[k]
        return True

q = SubseqQuery("abpcplea")
print([q.is_subsequence(t) for t in ["ale", "apple", "monkey", "plea", ""]])
rng = random.Random(152)
for _ in range(2000):
    s, d = gen(rng)
    q = SubseqQuery(s)
    assert all(q.is_subsequence(t) == is_subseq(t, s) for t in d)
print("SubseqQuery khớp hai con trỏ")

[True, True, False, True, True]


SubseqQuery khớp hai con trỏ


### 15.3. LeetCode 792: đếm số từ là dãy con của `s` (có tính từ lặp)

Chính là Cách 5 (xô chờ) nhưng **đếm** thay vì chọn đáp án. Đây là bài "kinh điển" của kỹ thuật xô.

In [40]:
def count_matching(s, words):
    waiting = defaultdict(list)
    for w in words:
        waiting[w[0]].append((w, 0))
    count = 0
    for ch in s:
        bucket = waiting.pop(ch, [])
        for w, p in bucket:
            p += 1
            if p == len(w):
                count += 1
            else:
                waiting[w[p]].append((w, p))
    return count

print(count_matching("abcde", ["a", "bb", "acd", "ace"]))      # LeetCode 792 ví dụ: 3
rng = random.Random(153)
for _ in range(2000):
    s, d = gen(rng)
    assert count_matching(s, d) == sum(is_subseq(w, s) for w in d)
print("count_matching khớp đếm trực tiếp")

3


count_matching khớp đếm trực tiếp


### 15.4. Các bài liên quan trên LeetCode (đọc thêm)

- **392. Is Subsequence:** đúng một cặp `(s, t)`; follow-up nhiều `t` → bảng kế tiếp hoặc `bisect`.
- **792. Number of Matching Subsequences:** xô chờ (như 15.3).
- **720. Longest Word in Dictionary:** từ phải được xây từng chữ cái một (mọi tiền tố đều có trong từ điển) — dùng set hoặc trie.
- **522. Longest Uncommon Subsequence II:** dùng lại phép kiểm tra dãy con giữa các chuỗi.
- **1143. Longest Common Subsequence:** dãy con chung dài nhất giữa hai chuỗi — quy hoạch động.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê dãy con | sinh mọi dãy con của `s`, tra từ điển | `O(2^m · m)` | `O(2^m · m)` | chỉ để hiểu đề / làm oracle với `m <= 14` |
| 2. Hai con trỏ từng từ | đảo vai trò, kiểm tra từng từ, giữ từ tốt nhất | `O(d · m)` | `O(1)` | **cách nộp bài chuẩn**, đơn giản nhất |
| 3. Sort `(-len, w)` | thứ tự ưu tiên ẩn trong khoá, dừng sớm | `O(d log d · L + d · m)` | `O(d)` | muốn code ngắn; đáp án thường là từ dài |
| 4. Bảng ký tự kế tiếp | `nxt[i][c]` nhảy cóc | `O(26 m + d · L)` | `O(26 m)` | một `s`, rất nhiều từ / truy vấn |
| 5. Xô chờ | quét `s` một lần cho mọi từ | `O(m + d · L)` | `O(d)` | `s` rất dài, từ điển rất lớn |

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [41]:
print(repr(Solution().findLongestWord("abcde", ["ace", "bd", "ae", "abd"])))
print(repr(Solution().findLongestWord("xyz", ["zy", "xz", "yx"])))
print(repr(Solution().findLongestWord("aaa", ["aaaa", "bb"])))
print(is_subseq("ace", "abcde"), is_subseq("aec", "abcde"))

'abd'
'xz'
''
True False


**Bài 2 (tính tay).** Với `s = "abcabc"` và từ `t = "cbc"`: ghép tham lam từng ký tự của `t`, ghi **chỉ số** trong `s` được ghép. Gán list chỉ số (ví dụ `[2, 4, 5]`) vào `ans2` (đang là `None`). Nếu không ghép được thì gán `[]`.

In [42]:
ans2 = None      # TODO: list chỉ số (hoặc [] nếu t không là dãy con)

**Bài 3 (điền TODO).** Hoàn thành `is_subsequence(t, s)` bằng hai con trỏ (trả về `True/False`; từ rỗng luôn là dãy con).

In [43]:
def is_subsequence(t, s):
    i = 0
    for ch in s:
        # TODO: nếu i chưa hết t và ch == t[i] thì i += 1
        pass
    # TODO: trả về kết quả
    return None

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_find_longest(s, dictionary)` theo **Cách 3**: sắp xếp theo khoá `(-độ dài, từ)` rồi trả về từ hợp lệ đầu tiên (dùng `is_subsequence` của bài 3, hoặc tự viết lại phép kiểm tra).

In [44]:
def my_find_longest(s, dictionary):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) `min_deletions(s, dictionary)`: số ký tự **ít nhất** phải xoá khỏi `s` để được **một từ nào đó** trong từ điển; trả về `-1` nếu không có từ nào tạo được. Ví dụ `min_deletions("abpcplea", ["ale","apple","monkey","plea"])` = 3 (giữ `apple`, xoá `b, c, a`). (b) `sorted_valid(s, dictionary)`: danh sách các từ **khác nhau** là dãy con của `s`, sắp theo thứ tự ưu tiên của đề (dài nhất trước, hoà thì từ điển nhỏ nhất trước).

In [45]:
def min_deletions(s, dictionary):
    # TODO
    pass

def sorted_valid(s, dictionary):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: với mỗi từ, thử ghép từng chữ theo thứ tự trong `s`; nhớ xét luật hoà.
- Bài 2: bắt đầu từ chỉ số 0, tìm `c` đầu tiên, rồi tìm `b` **sau** chỗ đó, rồi `c` sau chỗ đó.
- Bài 3: giống đoạn mã ở mục 3.2.
- Bài 4: `sorted(..., key=lambda w: (-len(w), w))`.
- Bài 5: số ký tự xoá = `len(s) - len(từ giữ lại)`; từ giữ lại nên là từ **dài nhất** hợp lệ.

**Mức 2 (vừa):**
- Bài 2: `c` ở chỉ số 2; `b` sau chỉ số 2 là chỉ số 4; `c` sau chỉ số 4 là chỉ số 5.
- Bài 3: `if i < len(t) and ch == t[i]: i += 1` và `return i == len(t)`.
- Bài 4: `for w in sorted(...): if is_subsequence(w, s): return w` rồi `return ""`.
- Bài 5(a): dùng `max` hoặc `Sol`-like: lấy `best = findLongestWord(s, dictionary)`; nếu `best == ""` thì `-1`, ngược lại `len(s) - len(best)`. Bài 5(b): `sorted({w for w in dictionary if ...}, key=lambda w: (-len(w), w))`.

**Mức 3 (gần đáp án):**
- Bài 3: xem `is_subseq` ở mục 3.2.
- Bài 4: xem Cách 3 ở mục 7.2.
- Bài 5(a): nhớ rằng đề 524 luôn trả về `""` khi không có, nên phải kiểm tra riêng (từ điển luôn có từ độ dài ≥ 1 nên `""` nghĩa là không có từ hợp lệ).

## 19. Đáp án tham khảo

In [46]:
# Bài 1: "abd"? -> kiểm tra từng dòng
assert Solution().findLongestWord("abcde", ["ace", "bd", "ae", "abd"]) == "abd"      # ace, abd cùng dài 3 -> abd < ace
assert Solution().findLongestWord("xyz", ["zy", "xz", "yx"]) == "xz"                 # chỉ xz hợp lệ
assert Solution().findLongestWord("aaa", ["aaaa", "bb"]) == ""
assert is_subseq("ace", "abcde") is True and is_subseq("aec", "abcde") is False
print("Bài 1 OK: 'abd', 'xz', '', True False")

# Bài 2
ans2_ref = [2, 4, 5]
assert match_positions("cbc", "abcabc") == ans2_ref
print("Bài 2 OK:", ans2_ref)

Bài 1 OK: 'abd', 'xz', '', True False
Bài 2 OK: [2, 4, 5]


In [47]:
# Bài 3
def is_subsequence_ref(t, s):
    i = 0
    for ch in s:
        if i < len(t) and ch == t[i]:
            i += 1
    return i == len(t)

rng = random.Random(1903)
for _ in range(2000):
    s, d = gen(rng)
    for w in d:
        assert is_subsequence_ref(w, s) == (w in all_subsequences(s))
assert is_subsequence_ref("", "abc") is True
print("Bài 3 OK")

# Bài 4
def my_find_longest_ref(s, dictionary):
    for w in sorted(dictionary, key=lambda w: (-len(w), w)):
        if is_subsequence_ref(w, s):
            return w
    return ""

rng = random.Random(1904)
for _ in range(2000):
    s, d = gen(rng)
    assert my_find_longest_ref(s, d) == oracle(s, d)
print("Bài 4 OK")

Bài 3 OK
Bài 4 OK


In [48]:
# Bài 5
def min_deletions_ref(s, dictionary):
    best = Solution().findLongestWord(s, dictionary)
    return -1 if best == "" else len(s) - len(best)

def sorted_valid_ref(s, dictionary):
    return sorted({w for w in dictionary if is_subsequence_ref(w, s)}, key=lambda w: (-len(w), w))

assert min_deletions_ref("abpcplea", ["ale", "apple", "monkey", "plea"]) == 3
assert min_deletions_ref("abc", ["d"]) == -1
assert sorted_valid_ref("abpcplea", ["ale", "apple", "monkey", "plea", "ale"]) == ["apple", "plea", "ale"]

def min_deletions_oracle(s, dictionary):
    subs = all_subsequences(s)
    lens = [len(s) - len(w) for w in set(dictionary) if w in subs]
    return min(lens) if lens else -1

rng = random.Random(1905)
for _ in range(2000):
    s, d = gen(rng)
    assert min_deletions_ref(s, d) == min_deletions_oracle(s, d)
    ref = sorted_valid_ref(s, d)
    subs = all_subsequences(s)
    assert ref == sorted({w for w in d if w in subs}, key=lambda w: (-len(w), w))
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template hai con trỏ cho từng từ (cách nộp):**

```python
class Solution:
    def findLongestWord(self, s, dictionary):
        def check(t, s):                    # t có là dãy con của s?
            i = 0
            for ch in s:
                if i < len(t) and ch == t[i]:
                    i += 1
            return i == len(t)

        ans = ""
        for t in dictionary:
            if check(t, s) and (len(t) > len(ans) or (len(t) == len(ans) and t < ans)):
                ans = t
        return ans
```

**Template sort + dừng sớm:**

```python
for w in sorted(dictionary, key=lambda w: (-len(w), w)):
    if check(w, s):
        return w
return ""
```

**Câu thần chú:** *đảo vai trò — kiểm tra từng từ là dãy con của `s`; so sánh bằng một khoá `(-len, w)`; ghép tham lam bằng hai con trỏ.*

**Checklist trước khi submit:**

- [ ] Dùng **dãy con** (hai con trỏ) chứ không phải `in` (chuỗi con) hay "đủ chữ cái"?
- [ ] Có xử lý luật hoà: cùng độ dài → từ nhỏ hơn theo từ điển (`t < ans`)?
- [ ] Không có từ hợp lệ → trả về `""` (và `ans` khởi tạo là `""`, không phải `None`)?
- [ ] Hai con trỏ có kiểm tra `i < len(t)` trước khi đọc `t[i]`?
- [ ] Từ dài hơn `s`, từ trùng `s`, ký tự lặp, từ trùng lặp trong từ điển đã thử?
- [ ] Độ phức tạp `O(d · m)` ≈ `10^6` cho giới hạn đề — đủ nhanh?